# Notebook 01 - Global Pair Screening (Hyperion / EMIT / EnMAP + Landsat 8/9)

**Member 1 - Satellite Data & Preprocessing**

| Part | Purpose | Status |
|---|---|---|
| 0 | Setup, config, asset check | run first |
| A | Derive land-cover profiles from rule-selected reference tiles | drafted, not yet run |
| B | Hyperion + EMIT + EnMAP footprint tables (EnMAP via DLR STAC) | EnMAP fields confirmed from pasted JSON; Hyperion/EMIT fields and EnMAP server-side filters **still to verify** |
| C | GEE screen: Landsat 8/9 match (+-15 d), QA_PIXEL cloud, WorldCover fractions | drafted, test on ~50 footprints first |
| D | Apply profiles, rank, shortlist | drafted |

**Honesty notes (put these in the report)**
- Profile *names* are the project's own terms. Reference tiles are selected by **rule** (Koppen-Geiger: Beck et al. 2018; GHSL Degree of Urbanisation settlement classes; WorldCover fractions: Zanaga et al. 2022), not by hand-picked cities.
- Rule thresholds (tile width 7.7 km = Hyperion swath; spacing 0.07°; 200 candidates and 60 kept tiles per profile; minimum tile settlement share 0.25–0.40; 10th–90th percentile ranges) are design parameters, not literature values. The achieved minimum settlement share of kept tiles (0.54–0.66) exceeded every threshold. The coast-distance rule is not applied.
- WorldCover is 2021: only a screening proxy for 2013-2017 Hyperion pairs. Confirm shortlisted pairs with a date-specific Landsat classification.
- Verify all asset IDs, class codes and API field names against current documentation before trusting results.

## Part 0 - Setup and configuration

In [ ]:
# !pip install -q earthaccess geopandas shapely requests   # uncomment on first run
import os, json, time, requests
import numpy as np, pandas as pd
import ee

from google.colab import drive
drive.mount("/content/drive")

BASE   = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"
CFG    = f"{BASE}/config"
OUT    = f"{BASE}/screening"
for d in (CFG, OUT): os.makedirs(d, exist_ok=True)

ee.Authenticate()          # first time only
ee.Initialize(project='hyperspectral-thermal-511303')            # add project="your-gcp-project" if required

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [ ]:
CONFIG = dict(
    # time window and matching
    start_date = "2013-04-01",
    end_date   = "2026-09-30",
    tolerance_days = 15,
    # sensors and their valid periods (footprint search windows)
    sensors = {
        "HYPERION": ("2013-04-01", "2017-03-31"),   # overlaps Landsat 8
        "EMIT":     ("2022-08-01", "2026-09-30"),   # overlaps Landsat 8 + 9
        # "ENMAP": add via CSV import in Part B
    },
    # screening filters
    min_clear_frac   = 0.85,   # clear fraction of Landsat QA_PIXEL inside the footprint
    # Part A sampling
    tile_width_m = 7_700,
    n_candidates = 200,        # random points per profile
    min_settlement_share = 0.5,
    sample_region = "India",   # reference tiles are sampled here; change to widen
    seed = 7,
)
json.dump(CONFIG, open(f"{CFG}/screen_config.json", "w"), indent=2)
CONFIG

{'start_date': '2013-04-01',
 'end_date': '2026-09-30',
 'tolerance_days': 15,
 'sensors': {'HYPERION': ('2013-04-01', '2017-03-31'),
  'EMIT': ('2022-08-01', '2026-09-30')},
 'min_clear_frac': 0.85,
 'tile_width_m': 7700,
 'n_candidates': 200,
 'min_settlement_share': 0.5,
 'sample_region': 'India',
 'seed': 7}

In [ ]:
# Check that the datasets exist in your Earth Engine account. Fix any MISSING id before continuing.
ASSETS = {
  "WorldCover":   "ESA/WorldCover/v200",
  "GHSL_SMOD":    "JRC/GHSL/P2023A/GHS_SMOD_V2-0",
  "Koppen_Beck":  "projects/hyperspectral-thermal-511303/assets/koppen_1991_2020",   # community catalog; verify id
  "Landsat8_L2":  "LANDSAT/LC08/C02/T1_L2",
  "Landsat9_L2":  "LANDSAT/LC09/C02/T1_L2",
  "GAUL0":        "FAO/GAUL/2015/level0",
}
for k, a in ASSETS.items():
    try:
        ee.data.getInfo(a); print("OK     ", k, a)
    except Exception as e:
        print("MISSING", k, a, "->", str(e)[:80])

OK      WorldCover ESA/WorldCover/v200
OK      GHSL_SMOD JRC/GHSL/P2023A/GHS_SMOD_V2-0
OK      Koppen_Beck projects/hyperspectral-thermal-511303/assets/koppen_1991_2020
OK      Landsat8_L2 LANDSAT/LC08/C02/T1_L2
OK      Landsat9_L2 LANDSAT/LC09/C02/T1_L2
OK      GAUL0 FAO/GAUL/2015/level0


In [ ]:
KOPPEN_ID = "projects/hyperspectral-thermal-511303/assets/koppen_1991_2020"
ASSETS["Koppen_Beck"] = KOPPEN_ID          # Part A reads this key

# Inspect band names and legends before relying on them
smod_ic = ee.ImageCollection(ASSETS["GHSL_SMOD"])
print("SMOD bands :", ee.Image(smod_ic.first()).bandNames().getInfo())
print("SMOD ids   :", [i["id"].split("/")[-1] for i in smod_ic.getInfo()["features"]][:10])

kg_img = ee.Image(KOPPEN_ID)
print("Koppen bands:", kg_img.bandNames().getInfo())
print("Koppen scale (m):", kg_img.projection().nominalScale().getInfo())

# Sanity check on class codes: which pixel values occur over Kerala?
kerala = ee.Geometry.Rectangle([75.0, 8.2, 77.3, 12.5])
print("Koppen values over Kerala:",
      kg_img.select(0).reduceRegion(ee.Reducer.frequencyHistogram(), kerala, 1000, maxPixels=1e9).getInfo())

SMOD bands : ['smod_code']
SMOD ids   : ['1975', '1980', '1985', '1990', '1995', '2000', '2005', '2010', '2015', '2020']
Koppen bands: ['b1']
Koppen scale (m): 1113.1949079327358
Koppen values over Kerala: {'b1': {'11': 91, '12': 2285.5176470588235, '15': 333.52156862745096, '2': 36252.56470588232, '3': 19503.286274509723, '6': 7299.149019607891}}


## Part A - Derive profiles from rule-selected reference tiles

| Profile | Koppen (Beck codes) | GHSL SMOD classes | Extra |
|---|---|---|---|
| humid_coastal | 1 Af, 2 Am, 3 Aw | 13 rural cluster, 21 suburban, 22 semi-dense cluster | within 50 km of coast (see note) |
| dense_city | any | 30 urban centre | - |
| semi_arid_city | 6 BSh, 7 BSk | 22, 23, 30 | - |

SMOD codes: 30 urban centre, 23 dense urban cluster, 22 semi-dense urban cluster, 21 suburban, 13 rural cluster, 12 low-density rural, 11 very low density.
**Check the Koppen legend and SMOD band name printed above before running.**

Coast rule: not applied in the sampling cell. After sampling, check kept `humid_coastal` tiles with a distance-to-coast layer, or review the tile map manually.

## Part A - Derive profiles from rule-selected reference tiles

Reference tiles (7.7 km) are sampled in the Kerala region with rules per profile. Their WorldCover class fractions (10th–90th percentile) define each profile's ranges, and the same tiles give each profile's minimum fit and settlement lower bound.

| Profile | Köppen gate (Beck codes) | Settlement rule (GHSL SMOD 2020) |
|---|---|---|
| humid_coastal | 1 Af, 2 Am, 3 Aw | settled share ≥ profile lower bound |
| dense_city | none (general urban profile) | settled share ≥ profile lower bound |
| semi_arid_city | 4 BWh, 5 BWk, 6 BSh, 7 BSk | settled share ≥ profile lower bound |

**settled** = share of SMOD classes 13 (rural cluster), 21 (suburban), 22 (semi-dense), 23 (dense cluster) and 30 (urban centre) in the tile or cell.

SMOD codes: 30 urban centre, 23 dense urban cluster, 22 semi-dense urban cluster, 21 suburban, 13 rural cluster, 12 low-density rural, 11 very low density.

**Check the Köppen legend and SMOD band name printed above before running.**

Coast rule: not applied in the sampling cell. After sampling, check the kept `humid_coastal` tiles with a distance-to-coast layer, or review the tile map manually.

In [ ]:
# ===== Part A: reference-tile sampling (replaces old Cell 7) =====
# Needs from earlier cells: ee, pd, CONFIG, CFG, ASSETS, KOPPEN_ID

# --- datasets ---
INDIA = (ee.FeatureCollection(ASSETS["GAUL0"])
           .filter(ee.Filter.eq("ADM0_NAME", CONFIG["sample_region"])).geometry())
KERALA = (ee.FeatureCollection("FAO/GAUL/2015/level1")
            .filter(ee.Filter.eq("ADM0_NAME", "India"))
            .filter(ee.Filter.eq("ADM1_NAME", "Kerala")).geometry())
# Dry belt (N. Africa - Middle East - S. Asia): semi-arid cities in India alone are cropland-dominated
DRY_BELT = ee.Geometry.Rectangle([-20, 5, 80, 40], geodesic=False)

smod = ee.Image(f"{ASSETS['GHSL_SMOD']}/2020").select("smod_code")   # GHSL SMOD 2020
wc   = ee.ImageCollection(ASSETS["WorldCover"]).first()               # ESA WorldCover 2021 (band "Map")
kg   = ee.Image(KOPPEN_ID).select(0)                                  # Koppen-Geiger 1991-2020

# --- rules (design parameters; state them in the report) ---
N_CAND, N_KEEP, SPACING_DEG = 200, 60, 0.07     # ~8 km spacing so tiles don't overlap
RULES = {
  # settled Kerala (rural cluster .. dense urban cluster); no forest requirement
  "humid_coastal":  dict(region=KERALA,   kg=None,         smod=[13, 21, 22, 23], min_share=0.30),
  # urban centre / dense urban cluster anywhere in India
  "dense_city":     dict(region=INDIA,    kg=None,         smod=[30, 23],         min_share=0.40),
  # arid/semi-arid Koppen (BWh, BWk, BSh, BSk) + settled cells, dry belt
  "semi_arid_city": dict(region=DRY_BELT, kg=[4, 5, 6, 7], smod=[21, 22, 23, 30], min_share=0.25),
}

def class_points(rule, n):
    cond = smod.remap(rule["smod"], [1]*len(rule["smod"]), 0).eq(1)
    if rule["kg"]:
        cond = cond.And(kg.remap(rule["kg"], [1]*len(rule["kg"]), 0).eq(1))
    return (cond.rename("cls").toByte()
            .stratifiedSample(numPoints=0, classBand="cls", region=rule["region"],
                              scale=1000, seed=CONFIG["seed"], classValues=[1],
                              classPoints=[n], geometries=True, tileScale=4))

def make_tile_fn(rule):
    in_class = smod.remap(rule["smod"], [1]*len(rule["smod"]), 0)
    def fn(p):
        box = p.geometry().buffer(CONFIG["tile_width_m"]/2).bounds()
        share = in_class.reduceRegion(ee.Reducer.mean(), box, 1000, maxPixels=1e9).values().get(0)
        h = wc.reduceRegion(ee.Reducer.frequencyHistogram(), box, 30, maxPixels=1e9).get("Map")
        c = p.geometry().coordinates()
        return ee.Feature(box, {"smod_share": share, "hist": h, "lon": c.get(0), "lat": c.get(1)})
    return fn

def thin(d, deg):
    keep = []
    for _, r in d.sort_values("smod_share", ascending=False).iterrows():
        if all(abs(r.lat - k.lat) > deg or abs(r.lon - k.lon) > deg for k in keep):
            keep.append(r)
    return pd.DataFrame(keep)

rows = []
for prof, rule in RULES.items():
    feats = class_points(rule, N_CAND).map(make_tile_fn(rule)).getInfo()["features"]
    if not feats:
        print(f"{prof:15s} NO candidates - check rule / Koppen codes"); continue
    d = pd.DataFrame([f["properties"] for f in feats])
    d["smod_share"] = d["smod_share"].fillna(0)
    d = d[d["hist"].notna()]
    n0 = len(d)
    d = d[d["smod_share"] >= rule["min_share"]]
    n1 = len(d)
    d = thin(d, SPACING_DEG).head(N_KEEP)
    d["profile"] = prof
    rows.append(d)
    print(f"{prof:15s} candidates {n0:4d} -> share filter {n1:4d} -> spaced {len(d):4d}")

tiles = pd.concat(rows, ignore_index=True)
tiles.to_csv(f"{CFG}/reference_tiles.csv", index=False)

/usr/local/lib/python3.13/dist-packages/ee/deprecation.py:215: DeprecationWarning: 

Attention required for FAO/GAUL/2015/level1! You are using a deprecated asset.
To make sure your code keeps working, please update it.
This dataset has been superseded by FAO/GAUL/2025/level1

Learn more: https://developers.google.com/earth-engine/datasets/catalog/FAO_GAUL_2015_level1

  warnings.warn(warning, category=DeprecationWarning)


humid_coastal   candidates  200 -> share filter  181 -> spaced   60
dense_city      candidates  200 -> share filter   93 -> spaced   60
semi_arid_city  candidates  200 -> share filter  158 -> spaced   60


In [ ]:
# Part A (continued): class-fraction profiles + settlement range, written to profiles_derived.json in one place
CL = {10:"tree",20:"shrub",30:"grass",40:"crop",50:"built",60:"bare",80:"water",90:"wetland",95:"mangrove"}
def fractions(h):
    h = {int(float(k)): v for k, v in h.items()}
    t = sum(h.values())
    return {CL[k]: v/t for k, v in h.items() if k in CL}

F = pd.DataFrame([fractions(h) for h in tiles["hist"]]).fillna(0)
CLS = ["tree","crop","built","water","bare","grass","shrub"]
for c in CLS:
    if c not in F: F[c] = 0.0
F["profile"] = tiles["profile"].values

profiles = {p: {c: [round(float(g[c].quantile(.10)),3), round(float(g[c].quantile(.90)),3)] for c in CLS}
            for p, g in F.groupby("profile")}

# settlement share of each reference tile (GHSL SMOD classes 13, 21, 22, 23, 30)
SETTLED = [13, 21, 22, 23, 30]
settled_img = smod.remap(SETTLED, [1]*len(SETTLED), 0)
pts = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([float(r.lon), float(r.lat)]), {"i": int(i)})
    for i, r in tiles.iterrows()])

def _settled(p):
    box = p.geometry().buffer(CONFIG["tile_width_m"]/2).bounds()
    s = settled_img.reduceRegion(ee.Reducer.mean(), box, 1000, maxPixels=1e9).values().get(0)
    return p.set({"settled": s})

res = pd.DataFrame([f["properties"] for f in pts.map(_settled).getInfo()["features"]]).set_index("i")
tiles["settled"] = res["settled"].reindex(tiles.index).fillna(0).values
for p, g in tiles.groupby("profile"):
    profiles[p]["settled"] = [round(float(g["settled"].quantile(.10)), 3),
                              round(float(g["settled"].quantile(.90)), 3)]

json.dump(profiles, open(f"{CFG}/profiles_derived.json", "w"), indent=2)
tiles.to_csv(f"{CFG}/reference_tiles.csv", index=False)

print("Median class fractions per profile")
print(F.groupby("profile")[CLS].median().round(2))
print("\nTiles per profile:\n", F["profile"].value_counts())
print("\nSettled range (10th-90th pct):", {p: profiles[p]["settled"] for p in profiles})

Median class fractions per profile
                tree  crop  built  water  bare  grass  shrub
profile                                                     
dense_city      0.26  0.31   0.23   0.01  0.01   0.03   0.00
humid_coastal   0.93  0.01   0.02   0.00  0.00   0.02   0.00
semi_arid_city  0.06  0.43   0.23   0.00  0.03   0.02   0.01

Tiles per profile:
 profile
humid_coastal     60
dense_city        60
semi_arid_city    60
Name: count, dtype: int64

Settled range (10th-90th pct): {'dense_city': [0.763, 1.0], 'humid_coastal': [0.739, 0.978], 'semi_arid_city': [0.686, 0.987]}


In [ ]:
print(tiles.groupby("profile")["smod_share"].describe()[["count","min","25%","50%","75%","max"]].round(2))
print()
for p, r in profiles.items():
    print(p, {c: r[c] for c in ["tree","crop","built","bare"]})

                count   min   25%   50%   75%   max
profile                                            
dense_city       60.0  0.54  0.62  0.74  0.89  1.00
humid_coastal    60.0  0.66  0.74  0.83  0.89  0.98
semi_arid_city   60.0  0.66  0.74  0.81  0.90  1.00

dense_city {'tree': [0.108, 0.465], 'crop': [0.027, 0.665], 'built': [0.045, 0.472], 'bare': [0.001, 0.029]}
humid_coastal {'tree': [0.768, 0.976], 'crop': [0.0, 0.103], 'built': [0.008, 0.032], 'bare': [0.0, 0.002]}
semi_arid_city {'tree': [0.012, 0.216], 'crop': [0.005, 0.818], 'built': [0.053, 0.612], 'bare': [0.002, 0.24]}


In [ ]:
#If the session restarts, you would not need to repeat Part B either. The merged footprints are saved, so run the Part 0 cells, the Köppen cell and Cell 7, and then load them with:


#import geopandas as gpd
#fp = gpd.read_file(f"{OUT}/footprints_all_sensors.geojson")
#fp["acq_datetime"] = pd.to_datetime(fp["acq_datetime"], utc=True)

## Part B - Hyperspectral footprint tables

Every adapter must output the same schema:

`sensor | scene_id | acq_datetime | geometry | cloud_pct | product_level | access_status`

### B1 - Hyperion (USGS M2M API)
Needs a free USGS ERS account and an **application token** (Account > Applications/Tokens). The dataset alias and field names below are my best recollection - print one raw record and adjust.

In [ ]:
import getpass
from shapely.geometry import shape
M2M = "https://m2m.cr.usgs.gov/api/api/json/stable/"

def m2m(endpoint, payload, key=None):
    h = {"X-Auth-Token": key} if key else {}
    r = requests.post(M2M + endpoint, json=payload, headers=h, timeout=120)
    r.raise_for_status()
    out = r.json()
    if out.get("errorCode"): raise RuntimeError(out)
    return out["data"]

user  = input("USGS username: ")
token = getpass.getpass("USGS M2M application token: ")
api_key = m2m("login-token", {"username": user, "token": token})

# Find the Hyperion dataset alias
ds = m2m("dataset-search", {"datasetName": "hyperion"}, api_key)
for d in ds: print(d["datasetAlias"], "|", d["collectionName"])

USGS username: <your-username>
USGS M2M application token: ··········
eo1_hyp_pub | EO-1 Hyperion
eo1_hyp_cloud_band_files_combined | EO-1 Hyperion Band Files


In [ ]:
HYP_ALIAS = "eo1_hyp_pub"        # set from the printed list above

def search_hyperion(t0, t1, bbox=(-180, -60, 180, 80), page=1000):
    """World search for Hyperion scenes; split the window by year to stay under result caps."""
    scenes, start = [], 1
    spatial = {"filterType": "mbr", "lowerLeft": {"latitude": bbox[1], "longitude": bbox[0]},
                                    "upperRight": {"latitude": bbox[3], "longitude": bbox[2]}}
    sf = {"filterType": "value", "filterId": None}  # placeholder if you add cloud filters
    while True:
        res = m2m("scene-search", {
            "datasetName": HYP_ALIAS, "maxResults": page, "startingNumber": start,
            "sceneFilter": {"spatialFilter": spatial,
                            "acquisitionFilter": {"start": t0, "end": t1}},
        }, api_key)
        scenes += res["results"]
        if len(res["results"]) < page: break
        start += page
    return scenes

hyp_raw = []
for y in range(2013, 2018):
    t0 = f"{y}-04-01" if y == 2013 else f"{y}-01-01"
    t1 = "2017-03-31" if y == 2017 else f"{y}-12-31"
    s = search_hyperion(t0, t1); print(y, len(s), "scenes"); hyp_raw += s

print(json.dumps(hyp_raw[0], indent=1)[:1500])   # inspect fields, then adjust the keys below

2013 4207 scenes
2014 5669 scenes
2015 5984 scenes
2016 2888 scenes
2017 396 scenes
{
 "browse": [
  {
   "id": "5e83a12e2901343f",
   "browseRotationEnabled": false,
   "browseName": "Standard Browse",
   "browsePath": "https://ims.cr.usgs.gov/browse/eo-1/hyp/7/14/2013/EO10070142013365110KZ_PF1_01.jpeg",
   "overlayPath": "https://ims.cr.usgs.gov/wms.py?service=WMS&request=GetMap&map=eo1-hyp&layers=EO1H0070142013365110KZ_PF1_01",
   "overlayType": "ims_wms",
   "thumbnailPath": "https://ims.cr.usgs.gov/thumbnail/eo-1/hyp/7/14/2013/EO10070142013365110KZ_PF1_01.jpeg"
  }
 ],
 "cloudCover": 20,
 "entityId": "EO1H0070142013365110KZ_PF1_01",
 "displayId": "EO1H0070142013365110KZ_PF1_01",
 "orderingId": null,
 "metadata": [
  {
   "id": "5e83a12ee61c7380",
   "fieldName": "Entity ID",
   "dictionaryLink": "https://www.usgs.gov/centers/eros/science/eo1-data-dictionary#entity_id",
   "value": "EO1H0070142013365110KZ_PF1_01"
  },
  {
   "id": "5e83a12e82077111",
   "fieldName": "Entity ID",
  

In [ ]:
def hyp_row(r):
    # Field names: verify against the printed record above.
    geom = shape(r["spatialCoverage"])                     # GeoJSON-like footprint
    dt   = r.get("temporalCoverage", {}).get("startDate") or r.get("publishDate")
    return dict(sensor="HYPERION", scene_id=r["entityId"], acq_datetime=pd.to_datetime(dt),
                geometry=geom, cloud_pct=r.get("cloudCover"), product_level="L1", access_status="open")

import geopandas as gpd
hyp = gpd.GeoDataFrame([hyp_row(r) for r in hyp_raw], geometry="geometry", crs="EPSG:4326")
print(len(hyp), "Hyperion footprints")
hyp.head()

19144 Hyperion footprints


,sensor,scene_id,acq_datetime,geometry,cloud_pct,product_level,access_status
0,HYPERION,EO1H0070142013365110KZ_PF1_01,2013-12-31 00:00:00-06:00,"POLYGON ((-52.17209 64.95842, -52.02391 64.935...",20,L1,open
1,HYPERION,EO1H0230082013365110KF_PF1_01,2013-12-31 00:00:00-06:00,"POLYGON ((-65.8596 72.66382, -65.66082 72.6318...",-1,L1,open
2,HYPERION,EO1H0250282013365110K3_PF1_01,2013-12-31 00:00:00-06:00,"POLYGON ((-90.5082 45.40736, -90.415 45.3934, ...",70,L1,open
3,HYPERION,EO1H0390352013365110KG_PF1_01,2013-12-31 00:00:00-06:00,"POLYGON ((-114.37336 35.75011, -114.28611 35.7...",20,L1,open
4,HYPERION,EO1H0730902013365110KV_TR2_01,2013-12-31 00:00:00-06:00,"POLYGON ((172.27113 -44.40566, 172.36984 -44.4...",70,L1,open


### B2 - EMIT (NASA Earthdata via `earthaccess`)

In [ ]:
!pip install -q earthaccess

In [ ]:
import earthaccess
earthaccess.login()                       # free NASA Earthdata account

t0, t1 = CONFIG["sensors"]["EMIT"]
emit_res = earthaccess.search_data(short_name="EMITL2ARFL",     # verify the current short name
                                   bounding_box=(-180, -52, 180, 52),
                                   temporal=(t0, t1))
print(len(emit_res), "EMIT granules")
print(json.dumps(emit_res[0]["umm"]["SpatialExtent"], indent=1)[:800])   # inspect once

Enter your Earthdata Login username: <your-username>
Enter your Earthdata password: ··········
301127 EMIT granules
{
 "HorizontalSpatialDomain": {
  "Geometry": {
   "GPolygons": [
    {
     "Boundary": {
      "Points": [
       {
        "Longitude": 73.69477844238281,
        "Latitude": 22.39096450805664
       },
       {
        "Longitude": 73.1546630859375,
        "Latitude": 21.768932342529297
       },
       {
        "Longitude": 73.73129272460938,
        "Latitude": 21.268230438232422
       },
       {
        "Longitude": 74.27140808105469,
        "Latitude": 21.890262603759766
       },
       {
        "Longitude": 73.69477844238281,
        "Latitude": 22.39096450805664
       }
      ]
     }
    }
   ]
  }
 }
}


In [ ]:
#def emit_row(g):
 #   umm = g["umm"]
  #  pts = umm["SpatialExtent"]["HorizontalSpatialDomain"]["Geometry"]["GPolygons"][0]["Boundary"]["Points"]
  #  ring = [(p["Longitude"], p["Latitude"]) for p in pts]
  #  from shapely.geometry import Polygon
  #  cc = next((a["Values"][0] for a in umm.get("AdditionalAttributes", []) if "cloud" in a["Name"].lower()), None)
  #  return dict(sensor="EMIT", scene_id=umm["GranuleUR"],
   #             acq_datetime=pd.to_datetime(umm["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]),
   #             geometry=Polygon(ring), cloud_pct=float(cc) if cc is not None else None,
   #             product_level="L2A", access_status="open")

#emit = gpd.GeoDataFrame([emit_row(g) for g in emit_res], geometry="geometry", crs="EPSG:4326")
#print(len(emit), "EMIT footprints")
#emit.head()

301127 EMIT footprints


,sensor,scene_id,acq_datetime,geometry,cloud_pct,product_level,access_status
0,EMIT,EMIT_L2A_RFL_001_20220810T034103_2222203_001,2022-08-10 03:41:03+00:00,"POLYGON ((73.69478 22.39096, 73.15466 21.76893...",None,L2A,open
1,EMIT,EMIT_L2A_RFL_002_20220810T034103,2022-08-10 03:41:03+00:00,"POLYGON ((74.27136 21.89029, 73.65487 22.34229...",None,L2A,open
2,EMIT,EMIT_L2A_RFL_001_20220810T034115_2222203_002,2022-08-10 03:41:15+00:00,"POLYGON ((74.20298 22.96522, 73.65519 22.34284...",None,L2A,open
3,EMIT,EMIT_L2A_RFL_002_20220810T034115,2022-08-10 03:41:15+00:00,"POLYGON ((74.77693 22.46003, 74.16009 22.9143,...",None,L2A,open
4,EMIT,EMIT_L2A_RFL_001_20220810T034127_2222203_003,2022-08-10 03:41:27+00:00,"POLYGON ((74.71402 23.53761, 74.16046 22.91481...",None,L2A,open


In [ ]:

from shapely.geometry import Polygon

def emit_row(g):
    umm = g["umm"]
    pts = umm["SpatialExtent"]["HorizontalSpatialDomain"]["Geometry"]["GPolygons"][0]["Boundary"]["Points"]
    ring = [(p["Longitude"], p["Latitude"]) for p in pts]
    cc = umm.get("CloudCover")
    return dict(sensor="EMIT", scene_id=umm["GranuleUR"],
                acq_datetime=pd.to_datetime(umm["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"]),
                geometry=Polygon(ring), cloud_pct=float(cc) if cc is not None else None,
                product_level="L2A", access_status="open")

emit = gpd.GeoDataFrame([emit_row(g) for g in emit_res], geometry="geometry", crs="EPSG:4326")
print(len(emit), "EMIT footprints;", emit["cloud_pct"].notna().mean().round(2), "have cloud values")

301127 EMIT footprints; 1.0 have cloud values


### B3 - EnMAP L2A (DLR EOC Geoservice STAC, automated)
Collection: `ENMAP_HSI_L2A` (surface reflectance over land, 224 bands, COG GeoTIFF, 30 m, one item = one ~30 km tile). Temporal extent starts 2022-04-27.

**Confirmed from the collection and item JSON you pasted:** property names `datetime`, `eo:cloud_cover`, `eo:snow_cover`, `enmap:overallQuality` (0 nominal, 1 reduced, 2 low), `enmap:cirrus_cover`, `enmap:haze_cover`, `enmap:cloud_shadow`, `version`; asset keys `image`, `quality_cloud`, `quality_cloud_shadow`, `quality_cirrus`, `quality_haze`, `quality_snow`, `quality_classes`, `quality_testflags`, `defective_pixel_mask`, `vnir`, `swir`, `metadata`; geometry is a GeoJSON polygon in EPSG:4326.

**Not confirmed:** whether the server honours the `datetime` and `bbox` query parameters (my test requests came back unfiltered, so I could not tell). The next cell tests this on a tiny request and stops with a clear message if the filters are ignored, because without them the full archive (very large JSON per item) would be paged.

Quality filters applied: `enmap:overallQuality == 0` and cloud cover <= `enmap_max_cloud` (default 30 %). Scene-level cloud is only a first screen; the footprint-level check in Part C and your quicklook review still apply.

In [ ]:
ENMAP_ITEMS = "https://geoservice.dlr.de/eoc/ogc/stac/v1/collections/ENMAP_HSI_L2A/items"
CONFIG.setdefault("enmap_max_cloud", 30)
CONFIG.setdefault("enmap_min_quality_ok", 0)       # keep only overallQuality <= this value
sess = requests.Session()

def stac_get(url, params=None, tries=4):
    for k in range(tries):
        try:
            r = sess.get(url, params=params, timeout=120, headers={"Accept": "application/geo+json"})
            r.raise_for_status()
            return r.json()
        except Exception as e:
            if k == tries - 1: raise
            time.sleep(3 * (k + 1))

def stac_pages(params, max_pages=500):
    """Yield items, following rel=next links."""
    page = stac_get(ENMAP_ITEMS, params)
    for _ in range(max_pages):
        for f in page.get("features", []):
            yield f
        nxt = next((l["href"] for l in page.get("links", []) if l.get("rel") == "next"), None)
        if not nxt: return
        page = stac_get(nxt)

# --- self-test: does the server honour datetime + bbox? (small request) ---
from shapely.geometry import box as _box, shape as _sh
t0, t1 = "2024-06-01T00:00:00Z", "2024-06-03T00:00:00Z"
bb = (68.0, 6.0, 98.0, 36.0)                        # India box; change to any region with known EnMAP coverage
test = stac_get(ENMAP_ITEMS, {"f": "application/geo+json", "limit": 5, "datetime": f"{t0}/{t1}",
                              "bbox": ",".join(map(str, bb))})
feats = test.get("features", [])
print("returned", len(feats), "items; numberMatched =", test.get("numberMatched"))
bad_t = [f["properties"]["datetime"] for f in feats
         if not (t0[:10] <= f["properties"]["datetime"][:10] <= t1[:10])]
bad_b = [f["id"] for f in feats if not _box(*bb).intersects(_sh(f["geometry"]))]
if bad_t or bad_b:
    raise RuntimeError(f"Server ignored filters (outside window: {len(bad_t)}, outside bbox: {len(bad_b)}). "
                       "Do NOT page the whole archive. Use pystac-client against the same STAC root "
                       "(https://geoservice.dlr.de/eoc/ogc/stac/v1) or the OpenSearch link on the collection page.")
print("datetime and bbox filters look honoured (0 items outside the test window/bbox; an empty result is also fine).")

returned 5 items; numberMatched = 5
datetime and bbox filters look honoured (0 items outside the test window/bbox; an empty result is also fine).


In [ ]:
from shapely.geometry import shape as _shape

def enmap_row(f):
    p, a = f["properties"], f.get("assets", {})
    return dict(sensor="ENMAP", scene_id=f["id"], acq_datetime=pd.to_datetime(p["datetime"], utc=True),
                geometry=_shape(f["geometry"]), cloud_pct=p.get("eo:cloud_cover"), product_level="L2A",
                access_status="open_cog", overall_quality=p.get("enmap:overallQuality"),
                proc_version=p.get("version"), asset_href=(a.get("image") or {}).get("href"),
                haze_pct=p.get("enmap:haze_cover"), cirrus_pct=p.get("enmap:cirrus_cover"),
                snow_pct=p.get("eo:snow_cover"), epsg=p.get("proj:epsg"))

def month_windows(start, end):
    s = pd.Timestamp(start, tz="UTC"); e = pd.Timestamp(end, tz="UTC")
    while s < e:
        n = min(s + pd.offsets.MonthBegin(1), e)
        yield s.strftime("%Y-%m-%dT%H:%M:%SZ"), n.strftime("%Y-%m-%dT%H:%M:%SZ")
        s = n

EN_START, EN_END = "2022-04-27", "2026-09-30"
ENMAP_BBOX = (-180, -60, 180, 80)        # world; narrow it if the run is too slow
rows = []
for w0, w1 in month_windows(EN_START, EN_END):
    n = 0
    for f in stac_pages({"f": "application/geo+json", "limit": 200, "datetime": f"{w0}/{w1}",
                         "bbox": ",".join(map(str, ENMAP_BBOX))}):
        r = enmap_row(f)
        if r["overall_quality"] is not None and r["overall_quality"] > CONFIG["enmap_min_quality_ok"]: continue
        if r["cloud_pct"] is not None and r["cloud_pct"] > CONFIG["enmap_max_cloud"]: continue
        rows.append(r); n += 1
    print(w0[:10], "kept", n, end="\r")

enmap = gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326")
enmap.to_file(f"{OUT}/enmap_footprints.geojson", driver="GeoJSON")      # cached; re-use instead of re-querying
print(len(enmap), "EnMAP L2A tiles kept (quality ok, cloud <=", CONFIG["enmap_max_cloud"], "%)")
enmap.head()

102920 EnMAP L2A tiles kept (quality ok, cloud <= 30 %)


,sensor,scene_id,acq_datetime,geometry,cloud_pct,product_level,access_status,overall_quality,proc_version,asset_href,haze_pct,cirrus_pct,snow_pct,epsg
0,ENMAP,ENMAP01-____L2A-DT0000000001_20220427T092911Z_...,2022-04-27 09:29:14.193552+00:00,"POLYGON ((28.46412 40.08973, 28.38508 39.81659...",0,L2A,open_cog,0,01.05.02,https://download.geoservice.dlr.de/ENMAP/files...,4,0,0,32635.0
1,ENMAP,ENMAP01-____L2A-DT0000000001_20220427T092907Z_...,2022-04-27 09:29:09.763183+00:00,"POLYGON ((28.54387 40.36022, 28.46268 40.08471...",0,L2A,open_cog,0,01.05.02,https://download.geoservice.dlr.de/ENMAP/files...,1,0,0,32635.0
2,ENMAP,ENMAP01-____L2A-DT0000000001_20220427T092903Z_...,2022-04-27 09:29:05.311089+00:00,"POLYGON ((28.62369 40.63062, 28.54256 40.35518...",0,L2A,open_cog,0,01.05.02,https://download.geoservice.dlr.de/ENMAP/files...,0,0,0,32635.0
3,ENMAP,ENMAP01-____L2A-DT0000000001_20220427T092858Z_...,2022-04-27 09:29:00.858996+00:00,"POLYGON ((28.70431 40.90082, 28.6222 40.6256, ...",0,L2A,open_cog,0,01.05.02,https://download.geoservice.dlr.de/ENMAP/files...,0,0,0,32635.0
4,ENMAP,ENMAP01-____L2A-DT0000000001_20220427T092854Z_...,2022-04-27 09:28:56.406896+00:00,"POLYGON ((28.7857 41.17091, 28.70281 40.89581,...",0,L2A,open_cog,0,01.05.02,https://download.geoservice.dlr.de/ENMAP/files...,6,0,0,32635.0


In [ ]:
# to avoid the LONG RUN TIME OF THE ENMAP ...WHILE RUUNNING TH ENOTEBOOKK AGAIN
#import geopandas as gpd
#enmap = gpd.read_file(f"{OUT}/enmap_footprints.geojson")
#enmap["acq_datetime"] = pd.to_datetime(enmap["acq_datetime"], utc=True)
#print(len(enmap), "EnMAP tiles loaded from file")

In [ ]:
KEEP = ["sensor","scene_id","acq_datetime","cloud_pct","product_level","access_status",
        "overall_quality","proc_version","asset_href","geometry"]
def std(g):
    if g is None or len(g) == 0: return None
    g = g.copy()
    for c in KEEP:
        if c not in g.columns: g[c] = None          # missing columns (e.g. cloud_pct) become null, not KeyError
    return g[KEEP]

parts = [x for x in (std(hyp), std(emit), std(enmap)) if x is not None]
fp = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), geometry="geometry", crs="EPSG:4326")
fp["acq_datetime"] = pd.to_datetime(fp["acq_datetime"], utc=True)
fp["cloud_pct"] = pd.to_numeric(fp["cloud_pct"], errors="coerce")
fp.loc[fp["cloud_pct"] < 0, "cloud_pct"] = np.nan      # Hyperion uses -1 for "unknown"
fp["t_ms"] = ((fp["acq_datetime"] - pd.Timestamp("1970-01-01", tz="UTC")) // pd.Timedelta(milliseconds=1)).astype("int64")
fp = fp[fp.is_valid & ~fp.is_empty].reset_index(drop=True)
fp.to_file(f"{OUT}/footprints_all_sensors.geojson", driver="GeoJSON")
print(fp["sensor"].value_counts())

sensor
EMIT        301127
ENMAP       102920
HYPERION     19144
Name: count, dtype: int64


In [ ]:
print(fp.groupby("sensor")["cloud_pct"].agg(["count", "min", "median", "max"]))

           count  min  median    max
sensor                              
EMIT      301127  0.0    66.0  100.0
ENMAP     102920  0.0     0.0   30.0
HYPERION   17888  0.0    20.0   90.0


## Part C - GEE screen
For each footprint: best Landsat 8/9 L2 scene within +-15 days (by clear fraction **inside the footprint**, from `QA_PIXEL`), WorldCover fractions inside the footprint, GHSL settlement share (SMOD classes 13, 21, 22, 23, 30) and the dominant Koppen-Geiger class (`kg_code`).

**Run on a small test first** (`TEST_N`, split equally across the three sensors) to confirm it works and to estimate time. Landsat QA_PIXEL bits: 1 dilated cloud, 2 cirrus, 3 cloud, 4 cloud shadow - verify in the Collection 2 documentation. Footprints are much larger than the 7.7 km reference tiles (median area: EMIT 7,312 km2, EnMAP 945 km2, Hyperion 817 km2), so whole-footprint fractions are diluted compared with the profiles.

## Part C - GEE screen latest ??????
For each footprint: best Landsat 8/9 L2 scene within +-15 days (by clear fraction **inside the footprint**, from `QA_PIXEL`), WorldCover fractions inside the footprint, GHSL settlement share (SMOD classes 13, 21, 22, 23, 30) and the dominant Koppen-Geiger class (`kg_code`). Footprints with no Landsat 8/9 scene in the window are dropped.

**Run on a small test first** (`TEST_N`, split equally across the three sensors) to confirm it works and to estimate time. Landsat QA_PIXEL bits: 1 dilated cloud, 2 cirrus, 3 cloud, 4 cloud shadow - verify in the Collection 2 documentation. Footprints are much larger than the 7.7 km reference tiles (median area: EMIT 7,312 km2, EnMAP 945 km2, Hyperion 817 km2), so whole-footprint fractions are diluted compared with the profiles.

In [ ]:
TEST_N = 51            # per-sensor sample of TEST_N/3; set to None for the full run
sub = fp if TEST_N is None else fp.groupby("sensor", group_keys=False).sample(n=TEST_N // 3, random_state=1)
sub = sub[["sensor","scene_id","t_ms","geometry"]].reset_index(drop=True)

# NB: very long footprint tables must be uploaded in chunks (about 2,000-5,000 features each)
fc = ee.FeatureCollection(json.loads(sub.to_json()))

TOL = CONFIG["tolerance_days"]
wc  = ee.ImageCollection(ASSETS["WorldCover"]).first()
LS  = ee.ImageCollection(ASSETS["Landsat8_L2"]).merge(ee.ImageCollection(ASSETS["Landsat9_L2"]))

def add_screen(f):
    g = f.geometry()
    t = ee.Date(ee.Number(f.get("t_ms")))
    cand = LS.filterBounds(g).filterDate(t.advance(-TOL, "day"), t.advance(TOL + 1, "day"))
    n = cand.size()

    def clear(img):
        qa  = img.select("QA_PIXEL")
        bad = qa.bitwiseAnd((1<<1)|(1<<2)|(1<<3)|(1<<4)).neq(0)
        frac = bad.Not().reduceRegion(ee.Reducer.mean(), g, 120, maxPixels=1e9).get("QA_PIXEL")
        dt = ee.Number(img.get("system:time_start")).subtract(t.millis()).abs().divide(86_400_000)
        return img.set({"clear_frac": ee.Number(frac), "dt_days": dt})

    def with_match():
        scored = cand.map(clear)
        best = ee.Image(scored.sort("clear_frac", False).first())
        hist = wc.reduceRegion(ee.Reducer.frequencyHistogram(), g, 30, maxPixels=1e9).get("Map")
        settled = settled_img.reduceRegion(ee.Reducer.mean(), g, 1000, maxPixels=1e9).values().get(0)
        kgc = kg.reduceRegion(ee.Reducer.mode(), g, 1000, maxPixels=1e9).values().get(0)
        return f.set({"n_landsat": n, "l_id": best.get("system:index"),
                      "clear_frac": best.get("clear_frac"), "dt_days": best.get("dt_days"),
                      "lc_hist": hist, "settled": settled, "kg_code": kgc})

    return ee.Feature(ee.Algorithms.If(n.gt(0), with_match(), f.set({"n_landsat": 0})))

out = fc.map(add_screen).filter(ee.Filter.gt("n_landsat", 0))
task = ee.batch.Export.table.toDrive(collection=out, description="pairs_screen_test",
                                     folder="MEMBER1___PREPROCESSING", fileFormat="CSV")
task.start()
print("Export started - monitor in the Earth Engine Tasks tab. The CSV lands in Drive/MEMBER1___PREPROCESSING/.")

Export started - monitor in the Earth Engine Tasks tab. The CSV lands in Drive/MEMBER1___PREPROCESSING/.


In [ ]:
# Poll the task
while task.active():
    print(task.status()["state"], end="\r"); time.sleep(15)
print(task.status())

{'state': 'COMPLETED', 'description': 'pairs_screen_test', 'priority': 100, 'creation_timestamp_ms': 1791975622024, 'update_timestamp_ms': 1791975639339, 'start_timestamp_ms': 1791975624294, 'task_type': 'EXPORT_FEATURES', 'destination_uris': ['<drive-folder>'], 'attempt': 1, 'batch_eecu_usage_seconds': 57.27412796020508, 'id': 'NNHV5HDADTZQ2GNEELXKMZHT', 'name': 'projects/hyperspectral-thermal-511303/operations/NNHV5HDADTZQ2GNEELXKMZHT'}


## Part D - Apply profiles, rank, shortlist
Moves the exported `pairs_screen.csv` into `screening/` first (or point the path to Drive).

In [ ]:
import glob, shutil
found = sorted(glob.glob("/content/drive/MyDrive/*/pairs_screen*.csv"), key=os.path.getmtime)
if found:
    print("moving", found[-1]); shutil.move(found[-1], f"{OUT}/pairs_screen.csv")
sc = pd.read_csv(f"{OUT}/pairs_screen.csv")

# hyperspectral scene cloud cover from Part B (EMIT/Hyperion/EnMAP), not the Landsat clear fraction
sc = sc.merge(fp[["scene_id", "cloud_pct"]].drop_duplicates("scene_id"), on="scene_id", how="left")
sc["cloud_pct"] = pd.to_numeric(sc["cloud_pct"], errors="coerce")

import re
def parse_hist(h):
    d = {int(float(k)): float(v) for k, v in re.findall(r"(\d+(?:\.\d+)?)\s*[=:]\s*([0-9.eE+-]+)", h)}
    t = sum(d.values())
    return {CL[k]: v/t for k, v in d.items() if k in CL} if t > 0 else {}

Fr = pd.DataFrame([parse_hist(h) if isinstance(h, str) else {} for h in sc["lc_hist"]]).fillna(0)
for c in CLS:
    if c not in Fr: Fr[c] = 0.0
sc = pd.concat([sc.drop(columns=["lc_hist"]), Fr[CLS]], axis=1)

profiles = json.load(open(f"{CFG}/profiles_derived.json"))

def fit(row, prof):
    ok, pen = 0, 0.0
    for c, (lo, hi) in profiles[prof].items():
        v = row[c]
        if lo <= v <= hi: ok += 1
        else: pen += min(abs(v-lo), abs(v-hi))
    return ok/len(profiles[prof]) - pen

for p in profiles:
    sc[f"fit_{p}"] = sc.apply(lambda r: fit(r, p), axis=1)

CLIMATE = {"humid_coastal": [1, 2, 3], "semi_arid_city": [4, 5, 6, 7], "dense_city": None}
MAX_CLOUD = 30

base = sc[(sc["clear_frac"] >= CONFIG["min_clear_frac"]) & (sc["dt_days"] <= CONFIG["tolerance_days"])
          & (sc["cloud_pct"].isna() | (sc["cloud_pct"] <= MAX_CLOUD))].copy()
for p in profiles:
    ok = base["settled"].ge(profiles[p]["settled"][0])
    if CLIMATE[p] is not None:
        ok &= base["kg_code"].isin(CLIMATE[p])
    base[f"score_{p}"] = base[f"fit_{p}"].where(ok)       # NaN where this profile's gates fail

cols = [f"score_{p}" for p in profiles]
good = base.dropna(subset=cols, how="all").copy()
good["best_profile"] = good[cols].idxmax(axis=1).str.replace("score_", "")
good["best_fit"]     = good[cols].max(axis=1)
good = good.sort_values("best_fit", ascending=False)
good.to_csv(f"{OUT}/ranked_pairs.csv", index=False)
print(len(sc), "screened ->", len(base), "pass Landsat/time/cloud ->", len(good), "pass settlement + climate")
print(good.groupby(["best_profile", "sensor"]).size())
good.groupby("best_profile").head(5)[["best_profile","sensor","scene_id","l_id","dt_days","clear_frac","cloud_pct","settled","kg_code","best_fit"]]

49 screened -> 29 pass Landsat/time/cloud -> 0 pass settlement + climate
Series([], dtype: int64)


,best_profile,sensor,scene_id,l_id,dt_days,clear_frac,cloud_pct,settled,kg_code,best_fit


In [ ]:
pre = sc[(sc["clear_frac"] >= CONFIG["min_clear_frac"]) & (sc["dt_days"] <= CONFIG["tolerance_days"])
         & (sc["cloud_pct"].isna() | (sc["cloud_pct"] <= MAX_CLOUD))].copy()
pre["best_profile"] = pre[[f"fit_{p}" for p in profiles]].idxmax(axis=1).str.replace("fit_", "")
pre["best_fit"]     = pre[[f"fit_{p}" for p in profiles]].max(axis=1)
pre["settled_ok"] = pre.apply(lambda r: pd.notna(r["settled"]) and r["settled"] >= profiles[r["best_profile"]]["settled"][0], axis=1)
pre["kg_ok"] = pre.apply(lambda r: CLIMATE[r["best_profile"]] is None or (pd.notna(r["kg_code"]) and int(r["kg_code"]) in CLIMATE[r["best_profile"]]), axis=1)
print(pre[["sensor","best_profile","settled","kg_code","best_fit","settled_ok","kg_ok"]].round(2).to_string())
print(pre[["settled_ok", "kg_ok"]].sum(), "of", len(pre))

      sensor    best_profile  settled  kg_code  best_fit  settled_ok  kg_ok
0       EMIT  semi_arid_city     0.01      6.0     -1.09       False   True
1       EMIT   humid_coastal     0.00      3.0     -0.34       False   True
3       EMIT  semi_arid_city     0.01      6.0     -0.72       False   True
8       EMIT  semi_arid_city     0.00      5.0     -1.07       False   True
16     ENMAP  semi_arid_city     0.00      4.0     -0.13       False   True
17     ENMAP  semi_arid_city     0.00      6.0     -1.05       False   True
18     ENMAP  semi_arid_city     0.00      NaN     -1.56       False  False
20     ENMAP   humid_coastal     0.00     26.0     -0.70       False  False
21     ENMAP      dense_city     0.00      6.0     -0.18       False   True
22     ENMAP  semi_arid_city     0.01      6.0     -1.24       False   True
23     ENMAP      dense_city     0.00     19.0     -0.75       False   True
24     ENMAP   humid_coastal     0.00     14.0     -0.02       False  False
25     ENMAP

##PART'E'

In [ ]:
GRID_DEG  = 0.07
PROJ_CRS  = "EPSG:4326"
TRANSFORM = [GRID_DEG, 0, -180, 0, -GRID_DEG, 90]
WORLD     = ee.Geometry.Rectangle([-180, -60, 180, 80], geodesic=False)
CLIMATE   = {"humid_coastal": [1, 2, 3], "semi_arid_city": [4, 5, 6, 7], "dense_city": None}

# settlement share and Koppen class aggregated to the 0.07-degree grid
settled_fixed = settled_img.setDefaultProjection(smod.projection())
settled_cell = (settled_fixed.reduceResolution(ee.Reducer.mean(), maxPixels=512)
                .reproject(PROJ_CRS, TRANSFORM).rename("settled"))
kg_cell = (kg.reduceResolution(ee.Reducer.mode(), maxPixels=1024)
           .reproject(PROJ_CRS, TRANSFORM).rename("kg"))
cells = ee.Image.cat([settled_cell, kg_cell])

def stage1(p):
    m = cells.select("settled").gte(profiles[p]["settled"][0])
    if CLIMATE[p]:
        m = m.And(cells.select("kg").remap(CLIMATE[p], [1]*len(CLIMATE[p]), 0).eq(1))
    fc = (cells.updateMask(m)
          .sample(region=WORLD, projection=ee.Projection(PROJ_CRS, TRANSFORM),
                  geometries=True, tileScale=8))
    return fc.map(lambda f: f.set({"profile": p,
                                   "lon": f.geometry().coordinates().get(0),
                                   "lat": f.geometry().coordinates().get(1)}))

stage1_fc = ee.FeatureCollection([stage1(p) for p in profiles]).flatten()
task = ee.batch.Export.table.toDrive(collection=stage1_fc, description="cells_stage1",
                                     folder="MEMBER1___PREPROCESSING", fileFormat="CSV",
                                     selectors=["profile", "settled", "kg", "lon", "lat"])
task.start()
print("Export started - cells_stage1.csv will appear in MEMBER1___PREPROCESSING")

Export started - cells_stage1.csv will appear in MEMBER1___PREPROCESSING


In [ ]:
c1 = pd.read_csv(f"{CFG}/../cells_stage1.csv") if os.path.exists(f"{CFG}/../cells_stage1.csv") else None
print(c1 is not None and c1.groupby("profile").size())

profile
dense_city        14968
humid_coastal      3827
semi_arid_city     2905
dtype: int64


#E-2

In [ ]:
import warnings
from shapely.geometry import box
MAX_CLOUD = 30
COV_MIN   = 0.5          # minimum share of the cell covered by the hyperspectral footprint
half = GRID_DEG / 2

c1 = pd.read_csv(f"{CFG}/../cells_stage1.csv")
cells_gdf = gpd.GeoDataFrame(
    c1, geometry=[box(x-half, y-half, x+half, y+half) for x, y in zip(c1["lon"], c1["lat"])],
    crs="EPSG:4326")
cells_gdf["cell_id"] = cells_gdf["lon"].round(4).astype(str) + "_" + cells_gdf["lat"].round(4).astype(str)

fp_ok = fp[fp["cloud_pct"].isna() | (fp["cloud_pct"] <= MAX_CLOUD)][
    ["sensor", "scene_id", "acq_datetime", "t_ms", "cloud_pct", "geometry"]]

parts = []
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    for s, g in fp_ok.groupby("sensor"):
        j = gpd.sjoin(g, cells_gdf[["cell_id", "profile", "settled", "kg", "lon", "lat", "geometry"]],
                      predicate="intersects", how="inner")
        cg = gpd.GeoSeries(cells_gdf.geometry.loc[j["index_right"]].values, index=j.index, crs="EPSG:4326")
        j["coverage"] = j.geometry.intersection(cg).area / cg.area
        j = j[j["coverage"] >= COV_MIN]
        print(s, len(g), "footprints ->", len(j), "footprint-cell pairs")
        parts.append(j.drop(columns=["geometry", "index_right"]))
pairs = pd.concat(parts, ignore_index=True)
pairs.to_csv(f"{OUT}/cell_footprint_pairs.csv", index=False)

print(pairs.groupby(["profile", "sensor"]).agg(pairs=("scene_id", "size"),
                                              cells=("cell_id", "nunique"),
                                              scenes=("scene_id", "nunique")))

EMIT 101301 footprints -> 93535 footprint-cell pairs
ENMAP 102920 footprints -> 28944 footprint-cell pairs
HYPERION 13412 footprints -> 4709 footprint-cell pairs
                         pairs  cells  scenes
profile        sensor                        
dense_city     EMIT      59020  11522    9024
               ENMAP     19341   8850    4863
               HYPERION   3089    609    1029
humid_coastal  EMIT       6259   2281     959
               ENMAP      3658   1623     759
               HYPERION   1241    212     345
semi_arid_city EMIT      28256   2873    5473
               ENMAP      5945   2191    1685
               HYPERION    379    120     232


#Step E3a
. For the candidate cells that sit inside a footprint, compute the WorldCover fractions. It saves results as it goes, so if the notebook disconnects, re-running the cell continues where it stopped instead of starting over. Add it as a new cell and run it:

In [ ]:
frac_path = f"{OUT}/cell_fractions.csv"
ALLC = list(CL.values())                       # tree, shrub, grass, crop, built, bare, water, wetland, mangrove

cells_u = pairs.drop_duplicates("cell_id")[["cell_id", "lon", "lat"]].reset_index(drop=True)
done_ids = set(pd.read_csv(frac_path)["cell_id"]) if os.path.exists(frac_path) else set()
todo = cells_u[~cells_u["cell_id"].isin(done_ids)]
print(len(cells_u), "unique cells;", len(done_ids), "already done;", len(todo), "to do")

wc_map = wc.select("Map")
CHUNK = 1000

def cell_fc(df):
    return ee.FeatureCollection([
        ee.Feature(ee.Geometry.Rectangle([r.lon-half, r.lat-half, r.lon+half, r.lat+half], geodesic=False),
                   {"cell_id": r.cell_id}) for r in df.itertuples()])

t0 = time.time()
for i in range(0, len(todo), CHUNK):
    part = todo.iloc[i:i+CHUNK]
    feats = wc_map.reduceRegions(cell_fc(part), ee.Reducer.frequencyHistogram(),
                                 scale=30, tileScale=4).getInfo()["features"]
    rows = []
    for f in feats:
        h = f["properties"].get("histogram")
        d = fractions(h) if h else {}
        d["cell_id"] = f["properties"]["cell_id"]
        rows.append(d)
    out_df = pd.DataFrame(rows).reindex(columns=["cell_id"] + ALLC).fillna(0)
    out_df.to_csv(frac_path, mode="a", header=not os.path.exists(frac_path), index=False)
    print(f"{min(i+CHUNK, len(todo))}/{len(todo)} cells done, {round((time.time()-t0)/60,1)} min", end="\r")
print("\nfinished")

14308 unique cells; 0 already done; 14308 to do
14308/14308 cells done, 6.1 min
finished


In [ ]:
# E3b-0: sanity check of the inputs for the fit calculation
import json, pandas as pd

frac = pd.read_csv(f"{OUT}/cell_fractions.csv")
print("fractions:", frac.shape, "| cells:", frac["cell_id"].nunique())
print("row-sum of class fractions (should be ~1):")
print(frac[ALLC].sum(axis=1).describe().round(3))

prof = json.load(open(f"{CFG}/profiles_derived.json"))
print("\nprofiles_derived.json top-level keys:", list(prof.keys()))
first = list(prof.keys())[0]
print(f"\nstructure of '{first}':")
print(json.dumps(prof[first], indent=1)[:1500])

ref = pd.read_csv(f"{CFG}/reference_tiles.csv")
print("\nreference_tiles.csv:", ref.shape)
print(list(ref.columns))
print(ref.head(3).T)

print("\ncells_gdf columns:", list(cells_gdf.columns))
print("pairs columns:", list(pairs.columns))

fractions: (14308, 10) | cells: 14308
row-sum of class fractions (should be ~1):
count    14308.000
mean         1.000
std          0.000
min          0.998
25%          1.000
50%          1.000
75%          1.000
max          1.000
dtype: float64

profiles_derived.json top-level keys: ['dense_city', 'humid_coastal', 'semi_arid_city']

structure of 'dense_city':
{
 "tree": [
  0.108,
  0.465
 ],
 "crop": [
  0.027,
  0.665
 ],
 "built": [
  0.045,
  0.472
 ],
 "water": [
  0.0,
  0.129
 ],
 "bare": [
  0.001,
  0.029
 ],
 "grass": [
  0.007,
  0.096
 ],
 "shrub": [
  0.0,
  0.091
 ],
 "settled": [
  0.763,
  1.0
 ]
}

reference_tiles.csv: (180, 6)
['hist', 'lat', 'lon', 'smod_share', 'profile', 'settled']
                                                            0  \
hist        {'10': 61404.03529411783, '30': 1115.270588235...   
lat                                                   9.73531   
lon                                                 76.528011   
smod_share               

#E-3B
Both files line up. The fractions sum to 1, and the profile ranges use the 7 WorldCover classes plus settled. Now E3b, which scores every candidate cell against its own profile.

In [ ]:
# E3b: fit of every candidate cell vs its profile; threshold from reference tiles
import ast, json, numpy as np, pandas as pd

prof = json.load(open(f"{CFG}/profiles_derived.json"))
ref  = pd.read_csv(f"{CFG}/reference_tiles.csv")
frac = pd.read_csv(f"{OUT}/cell_fractions.csv")

def fit_row(vals, rng):
    """vals: dict var->fraction; rng: dict var->[lo,hi].
    fit = share of variables inside range - total distance outside range."""
    inside, dist = 0, 0.0
    for v, (lo, hi) in rng.items():
        x = vals.get(v, 0.0)
        if lo <= x <= hi: inside += 1
        else: dist += (lo - x) if x < lo else (x - hi)
    return inside / len(rng) - dist

# 1) reference tiles: fractions from their class histograms
def hist_to_frac(h):
    h = ast.literal_eval(h) if isinstance(h, str) else h
    tot = sum(h.values())
    return {CL[int(k)]: v / tot for k, v in h.items() if int(k) in CL}

ref_fit = []
for _, r in ref.iterrows():
    vals = hist_to_frac(r["hist"]); vals["settled"] = r["settled"]
    ref_fit.append(fit_row(vals, prof[r["profile"]]))
ref["fit"] = ref_fit

# 2) data-derived minimum fit = 10th percentile of the reference tiles' own fit
thr = ref.groupby("profile")["fit"].quantile(0.10)
print("minimum fit per profile (10th pct of reference tiles):")
print(thr.round(3).to_string(), "\n")

# 3) candidate cells
cells = (pairs.drop_duplicates("cell_id")[["cell_id", "profile", "settled", "kg", "lon", "lat"]]
         .merge(frac, on="cell_id", how="inner"))
cells["fit"] = [fit_row({**{c: r[c] for c in ALLC}, "settled": r["settled"]}, prof[r["profile"]])
                for _, r in cells.iterrows()]
cells["fit_ok"] = cells["fit"] >= cells["profile"].map(thr)
cells["settled_ok"] = cells["settled"] >= cells["profile"].map({p: prof[p]["settled"][0] for p in prof})
cells["keep"] = cells["fit_ok"] & cells["settled_ok"]

print(cells.groupby("profile")[["fit_ok", "settled_ok", "keep"]].sum())
print("\ncells kept:", int(cells["keep"].sum()), "of", len(cells))
cells.to_csv(f"{OUT}/cell_fit.csv", index=False)

kept_pairs = pairs.merge(cells.loc[cells["keep"], ["cell_id", "fit"]], on="cell_id")
print("\nkept (cell, scene) pairs by profile x sensor:")
print(kept_pairs.groupby(["profile", "sensor"]).size().unstack(fill_value=0))

minimum fit per profile (10th pct of reference tiles):
profile
dense_city        0.306
humid_coastal     0.562
semi_arid_city    0.425 

                fit_ok  settled_ok  keep
profile                                 
dense_city        9603       10831  9603
humid_coastal      117        1562   117
semi_arid_city    1709        1915  1709

cells kept: 11429 of 14308

kept (cell, scene) pairs by profile x sensor:
sensor           EMIT  ENMAP  HYPERION
profile                               
dense_city      48221  15027      1850
humid_coastal    2306   1130       265
semi_arid_city  22683   4551       290


#E4
GEE can’t match 95,000 pairs in reasonable time, so E4 works on a subset:

Hyperion and humid-coastal: every pair is checked.
Other profile and sensor groups: the 1,500 lowest-cloud pairs per group.

That comes to about 11,800 pairs. For each pair, GEE finds every Landsat 8/9 scene within ±30 days and measures its clear fraction over the cell at 120 m. I use ±30 d so you can still choose the real tolerance (±15 or ±30) locally in E5 without re-running GEE. It’s resumable, so you can interrupt it and re-run safely.

In [ ]:
# E4: Landsat 8/9 matching + clear fraction per (cell, scene) pair  [resumable]
import os, json, pandas as pd, numpy as np, ee

HALF = 0.035
PER_GROUP = 1500
CHUNK = 200
out_path = f"{OUT}/landsat_match.csv"

# ---- 1) choose the subset of pairs to check -------------------------------
kp = kept_pairs.copy()
kp["cloud_sort"] = kp["cloud_pct"].fillna(50)          # unknown cloud -> middle
kp = kp.sort_values(["cloud_sort", "fit"], ascending=[True, False])
is_all = (kp["sensor"] == "HYPERION") | (kp["profile"] == "humid_coastal")
sub = pd.concat([kp[is_all],
                 kp[~is_all].groupby(["profile", "sensor"], group_keys=False).head(PER_GROUP)])
sub = sub.drop_duplicates(["scene_id", "cell_id"]).reset_index(drop=True)
sub["pair_key"] = sub["scene_id"].astype(str) + "|" + sub["cell_id"].astype(str)
print("pairs to check:", len(sub))
print(sub.groupby(["profile", "sensor"]).size().unstack(fill_value=0))

done = set(pd.read_csv(out_path)["pair_key"]) if os.path.exists(out_path) else set()
todo = sub[~sub["pair_key"].isin(done)].reset_index(drop=True)
print(f"{len(done)} already done; {len(todo)} to do")

# ---- 2) Landsat 8 + 9 collection with a clear-sky mask --------------------
L = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
       .merge(ee.ImageCollection("LANDSAT/LC09/C02/T1_L2"))
       .select("QA_PIXEL"))

def make_feat(r):
    geom = ee.Geometry.Rectangle([r.lon - HALF, r.lat - HALF, r.lon + HALF, r.lat + HALF], geodesic=False)
    return ee.Feature(geom, {"pair_key": r.pair_key, "t_ms": int(r.t_ms)})

def per_pair(f):
    f = ee.Feature(f); geom = f.geometry(); t = ee.Number(f.get("t_ms"))
    col = L.filterBounds(geom).filterDate(ee.Date(t).advance(-30, "day"),
                                          ee.Date(t).advance(30, "day").advance(1, "second"))
    def stat(im):
        clear = im.bitwiseAnd(30).eq(0)                  # bits 1-4: dilated cloud, cirrus, cloud, shadow
        fr = clear.unmask(0).reduceRegion(ee.Reducer.mean(), geom, 120, maxPixels=1e9).get("QA_PIXEL")
        return im.set({"clear": ee.Number(ee.Algorithms.If(fr, fr, 0)),
                       "dt": ee.Number(im.get("system:time_start")).subtract(t).divide(86400000)})
    c2 = col.map(stat)
    return f.set({"ls_ids": c2.aggregate_array("system:index"),
                  "ls_dt": c2.aggregate_array("dt"),
                  "ls_clear": c2.aggregate_array("clear")})

# ---- 3) run in chunks, append to CSV --------------------------------------
import time; t0 = time.time()
for i in range(0, len(todo), CHUNK):
    part = todo.iloc[i:i + CHUNK]
    fc = ee.FeatureCollection([make_feat(r) for r in part.itertuples()]).map(per_pair)
    feats = fc.select(["pair_key", "ls_ids", "ls_dt", "ls_clear"], retainGeometry=False).getInfo()["features"]
    rows = [{"pair_key": f["properties"]["pair_key"],
             "ls_ids": json.dumps(f["properties"].get("ls_ids", [])),
             "ls_dt": json.dumps(f["properties"].get("ls_dt", [])),
             "ls_clear": json.dumps(f["properties"].get("ls_clear", []))} for f in feats]
    pd.DataFrame(rows).to_csv(out_path, mode="a", header=not os.path.exists(out_path), index=False)
    print(f"{min(i + CHUNK, len(todo))}/{len(todo)} pairs done, {(time.time()-t0)/60:.1f} min")
sub.to_csv(f"{OUT}/landsat_match_subset.csv", index=False)
print("finished")

pairs to check: 10125
sensor          EMIT  ENMAP  HYPERION
profile                              
dense_city       956    999      1727
humid_coastal   2306   1130        38
semi_arid_city  1372   1372       225
0 already done; 10125 to do


200/10125 pairs done, 0.1 min
400/10125 pairs done, 0.3 min
600/10125 pairs done, 0.4 min


800/10125 pairs done, 0.7 min
1000/10125 pairs done, 0.9 min
1200/10125 pairs done, 1.0 min
1400/10125 pairs done, 1.2 min
1600/10125 pairs done, 1.3 min
1800/10125 pairs done, 1.4 min
2000/10125 pairs done, 1.5 min
2200/10125 pairs done, 2.3 min
2400/10125 pairs done, 2.4 min
2600/10125 pairs done, 2.4 min
2800/10125 pairs done, 2.6 min
3000/10125 pairs done, 2.7 min
3200/10125 pairs done, 2.8 min
3400/10125 pairs done, 2.8 min
3600/10125 pairs done, 2.9 min
3800/10125 pairs done, 3.1 min
4000/10125 pairs done, 3.3 min
4200/10125 pairs done, 3.4 min
4400/10125 pairs done, 3.4 min
4600/10125 pairs done, 3.5 min
4800/10125 pairs done, 3.7 min
5000/10125 pairs done, 3.8 min
5200/10125 pairs done, 3.9 min
5400/10125 pairs done, 3.9 min
5600/10125 pairs done, 4.0 min
5800/10125 pairs done, 4.2 min
6000/10125 pairs done, 4.3 min
6200/10125 pairs done, 4.5 min
6400/10125 pairs done, 4.7 min
6600/10125 pairs done, 4.8 min
6800/10125 pairs done, 4.9 min
7000/10125 pairs done, 5.1 min
7200/1012

E5 turns the Landsat matches into the shortlist.

In [ ]:
# E5: apply the pairing rules, rank, and write the shortlist
import json, numpy as np, pandas as pd

lm  = pd.read_csv(f"{OUT}/landsat_match.csv")
sub = pd.read_csv(f"{OUT}/landsat_match_subset.csv")
print("pairs checked:", len(lm), "of", len(sub))

# one row per (pair, Landsat scene)
lm["ls_ids"]   = lm["ls_ids"].apply(json.loads)
lm["ls_dt"]    = lm["ls_dt"].apply(json.loads)
lm["ls_clear"] = lm["ls_clear"].apply(json.loads)
ex = lm.explode(["ls_ids", "ls_dt", "ls_clear"]).dropna(subset=["ls_ids"]).copy()
ex["ls_dt"] = ex["ls_dt"].astype(float); ex["ls_clear"] = ex["ls_clear"].astype(float)
ex["abs_dt"] = ex["ls_dt"].abs()
ex["landsat"] = ex["ls_ids"].str[:4].map({"LC08": "L8", "LC09": "L9"})
ex = ex.merge(sub, on="pair_key", how="left")

MAX_CLOUD = 30                      # hyperspectral scene cloud %, same as screening
def shortlist(tol, min_clear, df=ex):
    d = df[(df["abs_dt"] <= tol) & (df["ls_clear"] >= min_clear)
           & (df["cloud_pct"].isna() | (df["cloud_pct"] <= MAX_CLOUD))]
    return (d.sort_values(["abs_dt", "ls_clear"], ascending=[True, False])
              .drop_duplicates("pair_key"))              # best Landsat per (scene, cell)

# sensitivity: how many pairs under each rule?
print("\nNumber of (scene, cell) pairs for each rule:")
rows = []
for tol in (15, 30):
    for mc in (0.85, 0.70, 0.50):
        s = shortlist(tol, mc)
        rows.append({"tolerance_d": tol, "min_clear": mc, "total": len(s),
                     "HYPERION": int((s.sensor == "HYPERION").sum()),
                     "EMIT": int((s.sensor == "EMIT").sum()),
                     "ENMAP": int((s.sensor == "ENMAP").sum()),
                     "unique_cells": s["cell_id"].nunique()})
print(pd.DataFrame(rows).to_string(index=False))

# main shortlist: the original design rules (±15 d, clear >= 0.85)
TOL, MIN_CLEAR = 15, 0.85
final = shortlist(TOL, MIN_CLEAR).copy()
order = {"HYPERION": 0, "EMIT": 1, "ENMAP": 2}
final["sensor_rank"] = final["sensor"].map(order)
final = final.sort_values(["sensor_rank", "fit", "abs_dt"], ascending=[True, False, True])
cols = ["sensor", "scene_id", "acq_datetime", "cloud_pct", "profile", "cell_id", "lon", "lat",
        "fit", "settled", "kg", "landsat", "ls_ids", "ls_dt", "ls_clear", "coverage"]
final[cols].to_csv(f"{OUT}/ranked_pairs.csv", index=False)

print(f"\nFINAL shortlist (±{TOL} d, clear >= {MIN_CLEAR}): {len(final)} pairs")
print(final.groupby(["profile", "sensor"]).size().unstack(fill_value=0))
print(final[["lat", "lon"]].round(1).drop_duplicates().shape[0], "distinct 0.1° locations")

pairs checked: 10125 of 10125

Number of (scene, cell) pairs for each rule:
 tolerance_d  min_clear  total  HYPERION  EMIT  ENMAP  unique_cells
          15       0.85   8990      1224  4385   3381          3346
          15       0.70   9242      1360  4471   3411          3404
          15       0.50   9479      1490  4541   3448          3447
          30       0.85   9573      1589  4519   3465          3454
          30       0.70   9766      1703  4582   3481          3497
          30       0.50   9909      1801  4612   3496          3521

FINAL shortlist (±15 d, clear >= 0.85): 8990 pairs
sensor          EMIT  ENMAP  HYPERION
profile                              
dense_city       940    961      1047
humid_coastal   2081   1060         6
semi_arid_city  1364   1360       171
2813 distinct 0.1° locations


# E6

In [ ]:
# E6: collapse to independent pairs (one best cell per hyperspectral scene) + Kerala-like check
r = pd.read_csv(f"{OUT}/ranked_pairs.csv")

# 1) how many independent scenes?
sc = r.groupby(["sensor", "profile"])["scene_id"].nunique().unstack(fill_value=0)
print("unique hyperspectral scenes by sensor x profile (±15 d, clear >= 0.85):")
print(sc, "\n")
print("unique scenes per sensor (any profile):")
print(r.groupby("sensor")["scene_id"].nunique(), "\n")

# 2) one best cell per scene: profile priority first (humid_coastal), then fit, then closeness in time
prio = {"humid_coastal": 0, "semi_arid_city": 1, "dense_city": 2}
r["prio"] = r["profile"].map(prio)
r["abs_dt"] = r["ls_dt"].astype(float).abs()
best = (r.sort_values(["prio", "fit", "abs_dt"], ascending=[True, False, True])
          .drop_duplicates("scene_id"))
order = {"HYPERION": 0, "EMIT": 1, "ENMAP": 2}
best["sensor_rank"] = best["sensor"].map(order)
best = best.sort_values(["sensor_rank", "prio", "fit"], ascending=[True, True, False])
best.drop(columns=["prio", "sensor_rank"]).to_csv(f"{OUT}/ranked_scenes.csv", index=False)

print("INDEPENDENT shortlist (one cell per scene):", len(best))
print(best.groupby(["profile", "sensor"]).size().unstack(fill_value=0), "\n")

# 3) the Kerala-like Hyperion candidates in detail
hk = best[(best.sensor == "HYPERION") & (best.profile == "humid_coastal")]
print("Hyperion + humid_coastal scenes:", len(hk))
print(hk[["scene_id", "acq_datetime", "cloud_pct", "lat", "lon", "fit", "kg", "ls_dt", "ls_clear"]].to_string(index=False))

# 4) where are the Hyperion scenes? (country-level rough check by lat/lon bands)
print("\nHyperion scenes by 5° latitude band:")
print(best[best.sensor == "HYPERION"].groupby((best.lat // 5) * 5).size().to_string())

unique hyperspectral scenes by sensor x profile (±15 d, clear >= 0.85):
profile   dense_city  humid_coastal  semi_arid_city
sensor                                             
EMIT             180            537             412
ENMAP            513            418             625
HYPERION         502              6             130 

unique scenes per sensor (any profile):
sensor
EMIT        1061
ENMAP       1469
HYPERION     621
Name: scene_id, dtype: int64 

INDEPENDENT shortlist (one cell per scene): 3151
sensor          EMIT  ENMAP  HYPERION
profile                              
dense_city       115    431       485
humid_coastal    537    418         6
semi_arid_city   409    620       130 

Hyperion + humid_coastal scenes: 6
                     scene_id              acq_datetime  cloud_pct    lat     lon      fit  kg     ls_dt  ls_clear
EO1H1160662016170110PF_AK3_01 2016-06-18 05:00:00+00:00        0.0 -8.385 115.365 0.857651   2  1.891347  0.978004
EO1H1160662014200110PF_PF1_01 2

## Handover to the next steps
`screening/ranked_pairs.csv` is the shortlist. For each accepted pair create `pairs/<sensor>_<scene_id>/` and record `sensor`, `region`, `dt_days`, `clear_frac`, class fractions and `profile` in its `metadata.json`.
Before accepting a pair, view its quicklook and check Hyperion/EMIT cloud and striping manually - metadata cloud is only a first filter.
Then continue to Notebook 02 (per-sensor branch: Hyperion needs your own atmospheric correction; EMIT/EnMAP L2A are already surface reflectance).\n\n**EnMAP fetch note:** each footprint row carries `asset_href` (spectral image COG). The quality layers (cloud, cloud shadow, cirrus, haze, snow, defective pixel mask, classes) are separate COGs in the item's assets; read them with a window over the AOI instead of downloading whole tiles. Record `proc_version` per pair - processor versions differ between scenes.

#E7

In [ ]:
# E7 (self-contained): handover list for Notebook 02 onward
from google.colab import drive
drive.mount("/content/drive")
import pandas as pd

ROOT = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"
OUT  = f"{ROOT}/screening"

r = pd.read_csv(f"{OUT}/ranked_pairs.csv")
r["abs_dt"] = r["ls_dt"].astype(float).abs()
r["prio"] = r["profile"].map({"humid_coastal": 0, "semi_arid_city": 1, "dense_city": 2})
r["loc"] = r["cell_id"]                                    # one 0.07° cell = one location

def pick_hyperion(df, cap=120, humid_dates=3):
    h = df[df.sensor == "HYPERION"].sort_values(["prio", "fit", "abs_dt"], ascending=[True, False, True])
    humid = h[h.profile == "humid_coastal"].groupby("loc", group_keys=False).head(humid_dates)
    rest  = h[h.profile != "humid_coastal"].drop_duplicates("loc")      # one date per location
    take  = cap - len(humid)
    semi  = rest[rest.profile == "semi_arid_city"].head(take // 2)
    dens  = rest[rest.profile == "dense_city"].head(take - len(semi))
    return pd.concat([humid, semi, dens])

hyp = pick_hyperion(r)
sec = (r[r.sensor != "HYPERION"]
         .sort_values(["prio", "fit", "abs_dt"], ascending=[True, False, True])
         .drop_duplicates("scene_id").groupby("sensor", group_keys=False).head(40))
handover = pd.concat([hyp, sec]).drop(columns=["prio", "loc", "abs_dt"])
handover.to_csv(f"{ROOT}/handover_pairs.csv", index=False)

print("handover pairs:", len(handover))
print(handover.groupby(["sensor", "profile"]).size().unstack(fill_value=0))
print("\nHyperion distinct locations:", hyp["loc"].nunique())

Mounted at /content/drive
handover pairs: 200
profile   dense_city  humid_coastal  semi_arid_city
sensor                                             
EMIT               0             40               0
ENMAP              0             40               0
HYPERION          58              4              58

Hyperion distinct locations: 118


In [ ]:
import os, glob
ROOT = "/content/drive/MyDrive/MEMBER1___PREPROCESSING"

hits = glob.glob(f"{ROOT}/**/legend*.txt", recursive=True)
print("legend files found:", hits)
for p in hits[:1]:
    print(open(p).read()[:1500])
if not hits:
    print("no legend.txt under the project folder")

legend files found: []
no legend.txt under the project folder


In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project="hyperspectral-thermal-511303")
kg = ee.Image("projects/hyperspectral-thermal-511303/assets/koppen_1991_2020").select("b1")

checks = {   # city: (lon, lat, expected Beck class, expected code)
    "Singapore":       (103.82,  1.35, "Af",  1),
    "Kochi (Kerala)":  ( 76.27,  9.93, "Am",  2),
    "Chennai":         ( 80.27, 13.08, "Aw",  3),
    "Dubai":           ( 55.30, 25.20, "BWh", 4),
    "Jaipur":          ( 75.79, 26.91, "BSh", 6),
    "Denver":          (-104.99, 39.74, "BSk", 7),
    "New York":        ( -74.00, 40.71, "Cfa", 14),
    "London":          (  -0.12, 51.51, "Cfb", 15),
}
fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([lo, la]), {"city": n})
                           for n, (lo, la, *_ ) in checks.items()])
res = kg.sampleRegions(fc, ["city"], scale=1000, geometries=False).getInfo()["features"]
got = {f["properties"]["city"]: f["properties"].get("b1") for f in res}
for n, (lo, la, cls, code) in checks.items():
    g = got.get(n)
    print(f"{n:15s} expected {cls:4s}={code:2d} | asset says {g} | {'OK' if g == code else 'CHECK'}")

Singapore       expected Af  = 1 | asset says 1 | OK
Kochi (Kerala)  expected Am  = 2 | asset says 2 | OK
Chennai         expected Aw  = 3 | asset says 3 | OK
Dubai           expected BWh = 4 | asset says 4 | OK
Jaipur          expected BSh = 6 | asset says 6 | OK
Denver          expected BSk = 7 | asset says 7 | OK
New York        expected Cfa =14 | asset says 14 | OK
London          expected Cfb =15 | asset says 15 | OK
